# Experiment: Large Model Calibration (Final Fix)
**Model:** `microsoft/deberta-v3-large`
**Status:** **RECOVERY MODE**

**Objective:**
To fix the stability issues observed in the previous "Large Model" run (where Validation Loss spiked to 0.94) and determine if the Large architecture can beat the Base model's high score.

**The Diagnosis & Fixes:**
1.  **Recall Collapse:** In the previous run, we removed the `1.5x` weight multiplier, which hurt Recall. We are **restoring the 1.5x Aggressive Weights** to force the model to pay attention to rare classes.
2.  **Overfitting:** The Large model memorized the training data too quickly. We are increasing `weight_decay` from `0.01` to `0.05` to force simpler decision boundaries.
3.  **Threshold Limits:** The previous run stopped searching for thresholds at `0.60`. We are expanding the search range to `0.95` because large models tend to be over-confident (calibrated probabilities close to 1.0), so the optimal cutoff might be higher (e.g., 0.80).

**Hypothesis:** By controlling the overfitting (Regularization) while simultaneously pushing the gradients (Weights), we can make the Large model behave as stably as the Base model but with higher capacity.

In [2]:
from google.colab import drive
import os
import shutil
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
import warnings


In [3]:
# 1. SETUP & INSTALLATION
# ---------------------------
print("⚙️ Installing libraries...")
# Using %%capture to hide the long output logs
!pip install transformers torch pandas scikit-learn tqdm sentencepiece accelerate -q

drive.mount('/content/drive')

# Set Seed for Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
warnings.filterwarnings('ignore')

⚙️ Installing libraries...
Mounted at /content/drive


In [3]:
# =========================================================
#  SUBTASK 3: ENGLISH ONLY - RECALL BOOSTER (DEBERTA LARGE)
#  Environment: Google Colab
# =========================================================


# 2. PATH CONFIGURATION & UNZIPPING
# ---------------------------
# We check if the folder exists. If not, we unzip it from Drive.
TARGET_DIR = 'subtask3' # The folder inside the zip
ZIP_PATH = '/content/drive/My Drive/NLP_AIMS/dev_phase.zip'

if not os.path.exists(TARGET_DIR):
    print(f"📂 Unzipping dataset from: {ZIP_PATH}")
    # We unzip into the current directory
    !unzip -o -q "{ZIP_PATH}"
else:
    print("✅ Dataset already extracted.")

# IMPORTANT: Based on your previous code, we assume the unzipped folder
# acts as the base. If the zip extracts into current dir, BASE_PATH is '.'
# If it extracts a folder named 'subtask3', we point there.
if os.path.exists('subtask3/train/eng.csv'):
    BASE_PATH = 'subtask3'
elif os.path.exists('train/eng.csv'):
    BASE_PATH = '.'
else:
    # Fallback search if folder structure is different
    # (Sometimes zips have extra nested folders)
    print("⚠️  Searching for 'eng.csv'...")
    for root, dirs, files in os.walk("."):
        if "eng.csv" in files and "train" in root:
            BASE_PATH = os.path.dirname(os.path.dirname(root))
            print(f"Found data at: {BASE_PATH}")
            break

print(f"📍 BASE_PATH set to: {BASE_PATH}")

# 3. MODEL CONFIGURATION (DEBERTA LARGE OPTIMIZED)
# ---------------------------
CONFIG = {
    'model': 'microsoft/deberta-v3-large',  # <--- LARGE MODEL
    'max_len': 256,
    'epochs': 5,
    'lr': 7e-6,            # <--- Low LR for stability
    'batch': 4,            # <--- Small Batch for Memory
    'grad_accum': 8,       # <--- Accumulate to simulate batch=32
    'warmup': 0.1,
    'weight_decay': 0.01,
    'augment_factor': 4,
    'label_smoothing': 0.05,
}

LABELS = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']

# 4. SYNONYMS & AUGMENTATION
# ---------------------------
SYNONYMS = {
    'stereotype': {'typical': ['usual', 'classic'], 'always': ['constantly', 'forever'], 'all': ['every', 'entire'],
                   'lazy': ['useless', 'idle'], 'stupid': ['dumb', 'idiotic'], 'criminal': ['thug', 'gangster']},
    'vilification': {'evil': ['vile', 'wicked'], 'scum': ['trash', 'garbage'], 'vermin': ['rats', 'pests'],
                     'monster': ['demon', 'devil'], 'worthless': ['pathetic', 'disgusting']},
    'dehumanization': {'animal': ['dog', 'beast'], 'subhuman': ['savage', 'primitive'],
                       'infestation': ['plague', 'disease'], 'breed': ['spawn', 'multiply']},
    'extreme_language': {'destroy': ['crush', 'wipe out'], 'exterminate': ['kill', 'eradicate'],
                         'massacre': ['genocide', 'bloodbath'], 'purge': ['cleanse', 'remove']},
    'lack_of_empathy': {'deserve': ['asked for it', 'karma'], 'fault': ['blame', 'own doing'],
                        'who cares': ['irrelevant', 'whatever'], 'deal with it': ['suck it up', 'cry more']},
    'invalidation': {'exaggerate': ['drama queen', 'overreacting'], 'fake': ['hoax', 'staged'],
                     'lie': ['bullshit', 'myth'], 'pretending': ['acting', 'playing victim']}
}

def augment_text(text, category, prob=0.35):
    cat_syns = SYNONYMS.get(category, {})
    if not cat_syns: return text
    words = text.split()
    aug_words = []
    for word in words:
        w_lower = word.lower().strip('.,!?;:')
        if w_lower in cat_syns and random.random() < prob:
            syn = random.choice(cat_syns[w_lower])
            if word[0].isupper(): syn = syn.capitalize()
            aug_words.append(syn)
        else:
            aug_words.append(word)
    return ' '.join(aug_words)

def augment_df(df, factor=3):
    print(f"Augmenting English Data: {len(df)} samples")
    aug_texts, aug_labels = [], []
    for _, row in df.iterrows():
        text = row['text']
        label_vals = {label: row[label] for label in LABELS}
        pos_labels = [l for l, v in label_vals.items() if v == 1]

        if pos_labels:
            for _ in range(factor):
                cat = random.choice(pos_labels)
                aug_text = augment_text(text, cat)
                if aug_text != text:
                    aug_texts.append(aug_text)
                    aug_labels.append(label_vals.copy())

    if aug_texts:
        aug_df = pd.DataFrame({'text': aug_texts, **{label: [d[label] for d in aug_labels] for label in LABELS}})
        result = pd.concat([df, aug_df], ignore_index=True)
        print(f"  -> Added {len(aug_df)} new samples. Total: {len(result)}")
        return result
    return df

# 5. CLASSES: DATASET & TRAINER
# ---------------------------
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Move weights to same device as logits
        weights = self.class_weights.to(logits.device)
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

class PolarizationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts; self.labels = labels; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=self.max_len, padding="max_length", return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

def compute_metrics(pred):
    probs = torch.sigmoid(torch.from_numpy(pred.predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    return {'f1_macro': f1_score(pred.label_ids, preds, average='macro', zero_division=0)}

# 6. MAIN PIPELINE
# ---------------------------
# =========================================================
#  FINAL FIX: UNLOCKED THRESHOLDS + HIGH RECALL WEIGHTS
# =========================================================

def train_english_optimized():
    print("="*60)
    print("🚀 STARTING DEBERTA-V3-LARGE (FINAL FIX)")
    print("="*60)

    # A. Load & Augment
    train_file = os.path.join(BASE_PATH, "train/eng.csv")
    if not os.path.exists(train_file):
        raise FileNotFoundError(f"Could not find training file at: {train_file}")

    df_raw = pd.read_csv(train_file)
    df_aug = augment_df(df_raw, factor=CONFIG['augment_factor'])

    # B. Split
    train_df, val_df = train_test_split(df_aug, test_size=0.15, random_state=42)
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")

    # C. Calculate Class Weights (RESTORED TO 1.5)
    # We need this aggression to keep Recall high (0.74)
    pos_counts = train_df[LABELS].sum()
    neg_counts = len(train_df) - pos_counts
    weights = (neg_counts / (pos_counts + 1e-6)) * 1.5
    class_weights = torch.tensor(weights.values, dtype=torch.float)
    print(f"Class Weights: {class_weights.tolist()}")

    # D. Model & Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model'])
    model = AutoModelForSequenceClassification.from_pretrained(CONFIG['model'], num_labels=6, problem_type="multi_label_classification")

    model.config.use_cache = False # Required for gradient checkpointing

    train_ds = PolarizationDataset(train_df['text'].tolist(), train_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])
    val_ds = PolarizationDataset(val_df['text'].tolist(), val_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])

    args = TrainingArguments(
        output_dir="./results_deberta_large_final",
        num_train_epochs=CONFIG['epochs'],
        learning_rate=CONFIG['lr'],
        per_device_train_batch_size=CONFIG['batch'],
        gradient_accumulation_steps=CONFIG['grad_accum'],
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        fp16=True,
        warmup_ratio=CONFIG['warmup'],
        weight_decay=0.05, # Increased slightly to reduce overfitting (Val loss 0.94)
        eval_strategy="steps", eval_steps=100,
        save_strategy="steps", save_steps=100,
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        report_to="none", save_total_limit=1,
        logging_steps=50
    )

    trainer = WeightedLossTrainer(
        class_weights=class_weights,
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
    )

    trainer.train()

    # E. THRESHOLD OPTIMIZATION (THE FIX)
    print("\n⚖️  OPTIMIZING THRESHOLDS (Range Expanded: 0.15 - 0.95)...")
    val_preds = trainer.predict(val_ds)
    val_probs = torch.sigmoid(torch.tensor(val_preds.predictions)).numpy()

    best_thresholds = []
    print(f"{'Label':<20} | {'Best Thresh':<12} | {'F1':<8} | {'Prec':<8} | {'Recall':<8}")
    print("-" * 70)

    for i, label in enumerate(LABELS):
        best_t, best_f1, best_p, best_r = 0.5, 0, 0, 0

        # 🔴 CHANGED: Search up to 0.95 instead of stopping at 0.60
        for t in np.arange(0.15, 0.95, 0.01):
            preds = (val_probs[:, i] > t).astype(int)
            f1 = f1_score(val_df[LABELS].values[:, i], preds, zero_division=0)

            # Prefer higher threshold if F1 is equal (boosts precision)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
                best_p = precision_score(val_df[LABELS].values[:, i], preds, zero_division=0)
                best_r = recall_score(val_df[LABELS].values[:, i], preds, zero_division=0)

        best_thresholds.append(best_t)
        print(f"{label:<20} | {best_t:.2f}         | {best_f1:.4f}   | {best_p:.4f}   | {best_r:.4f}")

    # F. Predict on Dev
    print("\n🔮 PREDICTING ON DEV SET...")
    dev_file = os.path.join(BASE_PATH, "dev/eng.csv")
    if not os.path.exists(dev_file):
         raise FileNotFoundError(f"Could not find dev file at: {dev_file}")

    dev_df = pd.read_csv(dev_file)
    dev_ds = PolarizationDataset(dev_df['text'].tolist(), [[0]*6]*len(dev_df), tokenizer, CONFIG['max_len'])

    dev_preds = trainer.predict(dev_ds)
    dev_probs = torch.sigmoid(torch.tensor(dev_preds.predictions)).numpy()

    final_preds = (dev_probs > np.array(best_thresholds)).astype(int)

    sub_df = pd.DataFrame(final_preds, columns=LABELS)
    sub_df.insert(0, 'id', dev_df['id'])
    sub_df.to_csv("submission_deberta_large_fixed.csv", index=False)

    print("\n DONE! File saved: submission_deberta_large_fixed.csv")

if __name__ == "__main__":
    train_english_optimized()

📂 Unzipping dataset from: /content/drive/My Drive/NLP_AIMS/dev_phase.zip
📍 BASE_PATH set to: subtask3
🚀 STARTING DEBERTA-V3-LARGE (FINAL FIX)
Augmenting English Data: 3222 samples
  -> Added 49 new samples. Total: 3271
Train: 2780 | Val: 491
Class Weights: [7.998860836029053, 3.9367666244506836, 10.213482856750488, 4.5260114669799805, 11.450310707092285, 6.250929355621338]


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

...

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1 Macro
100,1.298100,1.159899,0.318465
200,1.069000,0.942520,0.436591
300,0.882600,0.894510,0.464739
400,0.827100,0.933566,0.490403



⚖️  OPTIMIZING THRESHOLDS (Range Expanded: 0.15 - 0.95)...


Label                | Best Thresh  | F1       | Prec     | Recall  
----------------------------------------------------------------------
stereotype           | 0.86         | 0.5974   | 0.5679   | 0.6301
vilification         | 0.78         | 0.7165   | 0.6791   | 0.7583
dehumanization       | 0.77         | 0.4146   | 0.2857   | 0.7556
extreme_language     | 0.60         | 0.6284   | 0.5125   | 0.8119
lack_of_empathy      | 0.40         | 0.3486   | 0.2222   | 0.8085
invalidation         | 0.78         | 0.4712   | 0.3603   | 0.6806

🔮 PREDICTING ON DEV SET...



 DONE! File saved: submission_deberta_large_fixed.csv


### 🔧 Experiment Conclusion: Capacity vs. Data Size

**Outcome:** **Did NOT beat the Base Model High Score (0.508)**

**Final Analysis:**
Despite the fixes (High Regularization, Aggressive Weights, Expanded Thresholds), the Large model essentially matched the Base model's performance but took 3x longer to train.
* **The "Data Wall":** With only ~3,000 samples, we have hit the "Data Wall." The dataset is simply too small to justify the 304M parameters of `DeBERTa-Large`. The extra parameters are likely fitting noise rather than generalized signal.
* **Optimization Difficulty:** Large models are notoriously hard to fine-tune on small batches (Batch Size 4). The noise in the gradients likely prevented the model from finding the global optimum that the Base model found easily with Batch Size 16.

**Strategic Decision:**
We formally retire the `Large` architecture for this task. The **Champion Model remains `DeBERTa-v3-Base`** due to its efficiency and superior generalization on this specific dataset size.

# Experiment: The "Specialist Ensemble"
**Strategy:** **Hybrid Architecture (RoBERTa + DeBERTa)**
**Status:** **EXPERIMENTAL (Over-Engineering Check)**

**Objective:**
To beat the high score of `0.508` by combining two distinct models, each playing to its strengths:
1.  **The Generalist:** Your best `DeBERTa-v3-Base` model (which understands context well).
2.  **The Specialist:** `twitter-roberta-base-hate-latest` (which understands hate speech keywords well).

**The Logic:**
* `DeBERTa` is great at "Lack of Empathy" and "Invalidation" (contextual).
* `RoBERTa-Hate` is pre-trained on offensive tweets, so it should be better at "Vilification" and "Extreme Language" (explicit).
* **Union Strategy:** We use a logical `OR` for hard classes. If *either* the Generalist predicts `DeBERTa` OR the Specialist is highly confident (`> 0.40`), we flag it as positive. This is a "Recall Boosting" hack.

**Hypothesis:** An ensemble will correct the blind spots of the individual models.

In [9]:
# ==================================================================================
#  🏆 FINAL PIPELINE: USER'S BEST BASE + SPECIALIST ENSEMBLE (FIXED)
#  1. Twitter-RoBERTa (Specialist) -> Targeted for Dehumanization/Empathy
#  2. User's DeBERTa-Base (Generalist) -> Your best performing config
#  3. Ensemble -> Smart Combination
# ==================================================================================

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from google.colab import drive
import warnings

warnings.filterwarnings("ignore")

# =========================================================
#  0. SETUP & DATA PREP
# =========================================================
print("⚙️  SETTING UP ENVIRONMENT...")
!pip install transformers torch pandas scikit-learn tqdm sentencepiece accelerate -q

drive.mount('/content/drive')

# Handle Zip/Paths
ZIP_PATH = '/content/drive/My Drive/data/NLP_AIMS/dev_phase.zip'
TARGET_DIR = 'subtask3'

if not os.path.exists(TARGET_DIR):
    if os.path.exists(ZIP_PATH):
        print("📂 Unzipping dataset...")
        !unzip -o -q "{ZIP_PATH}"
    else:
        print(f"⚠️ Zip file not found at {ZIP_PATH}. Assuming data is already present.")

if os.path.exists('subtask3/train/eng.csv'): BASE_PATH = 'subtask3'
elif os.path.exists('train/eng.csv'): BASE_PATH = '.'
else: BASE_PATH = '.'

LABELS = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Shared Classes
class PolarizationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts; self.labels = labels; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=self.max_len, padding="max_length", return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

def compute_metrics(pred):
    probs = torch.sigmoid(torch.from_numpy(pred.predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    return {'f1_macro': f1_score(pred.label_ids, preds, average='macro', zero_division=0)}

# =========================================================
#  1. TRAIN THE SPECIALIST (Twitter-RoBERTa-Hate)
# =========================================================
def train_specialist():
    print("\n" + "="*60)
    print("🚀 STEP 1: TRAINING SPECIALIST (Twitter-RoBERTa)")
    print("="*60)

    CONFIG = {
        'model': 'cardiffnlp/twitter-roberta-base-hate-latest',
        'max_len': 128, 'epochs': 6, 'lr': 2e-5, 'batch': 32
    }

    df = pd.read_csv(os.path.join(BASE_PATH, "train/eng.csv"))
    train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model'])

    # 🔴 FIX APPLIED HERE: ignore_mismatched_sizes=True
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model'],
        num_labels=6,
        problem_type="multi_label_classification",
        ignore_mismatched_sizes=True # <--- CRITICAL FIX
    )

    train_ds = PolarizationDataset(train_df['text'].tolist(), train_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])
    val_ds = PolarizationDataset(val_df['text'].tolist(), val_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])

    args = TrainingArguments(
        output_dir="./results_specialist",
        num_train_epochs=CONFIG['epochs'], learning_rate=CONFIG['lr'],
        per_device_train_batch_size=CONFIG['batch'], gradient_accumulation_steps=1,
        fp16=True, eval_strategy="steps", eval_steps=100, save_strategy="steps", save_steps=100,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", save_total_limit=1, report_to="none"
    )

    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    trainer.train()

    # Predict on Dev
    print("🔮 Specialist Predicting on Dev Set...")
    dev_df = pd.read_csv(os.path.join(BASE_PATH, "dev/eng.csv"))
    dev_ds = PolarizationDataset(dev_df['text'].tolist(), [[0]*6]*len(dev_df), tokenizer, CONFIG['max_len'])
    preds = trainer.predict(dev_ds)
    probs = torch.sigmoid(torch.tensor(preds.predictions)).numpy()

    # Return probabilities for ensemble
    return pd.DataFrame(probs, columns=LABELS), dev_df['id']

# =========================================================
#  2. TRAIN YOUR BEST BASE MODEL (User's Exact Config)
# =========================================================
def train_users_best_base():
    print("\n" + "="*60)
    print("🚀 STEP 2: TRAINING YOUR BEST DEBERTA-BASE CODE")
    print("="*60)

    # USER'S EXACT CONFIG
    CONFIG = {
        'model': 'microsoft/deberta-v3-base',
        'max_len': 256, 'epochs': 6, 'lr': 2e-5, 'batch': 16,
        'grad_accum': 2, 'warmup': 0.1, 'weight_decay': 0.01,
        'augment_factor': 4, # High Augmentation
        'label_smoothing': 0.05,
    }

    # USER'S SYNONYMS & AUGMENTATION LOGIC
    SYNONYMS = {
        'stereotype': {'typical': ['usual', 'classic'], 'always': ['constantly', 'forever'], 'all': ['every', 'entire'],
                       'lazy': ['useless', 'idle'], 'stupid': ['dumb', 'idiotic'], 'criminal': ['thug', 'gangster']},
        'vilification': {'evil': ['vile', 'wicked'], 'scum': ['trash', 'garbage'], 'vermin': ['rats', 'pests'],
                         'monster': ['demon', 'devil'], 'worthless': ['pathetic', 'disgusting']},
        'dehumanization': {'animal': ['dog', 'beast'], 'subhuman': ['savage', 'primitive'],
                           'infestation': ['plague', 'disease'], 'breed': ['spawn', 'multiply']},
        'extreme_language': {'destroy': ['crush', 'wipe out'], 'exterminate': ['kill', 'eradicate'],
                             'massacre': ['genocide', 'bloodbath'], 'purge': ['cleanse', 'remove']},
        'lack_of_empathy': {'deserve': ['asked for it', 'karma'], 'fault': ['blame', 'own doing'],
                            'who cares': ['irrelevant', 'whatever'], 'deal with it': ['suck it up', 'cry more']},
        'invalidation': {'exaggerate': ['drama queen', 'overreacting'], 'fake': ['hoax', 'staged'],
                         'lie': ['bullshit', 'myth'], 'pretending': ['acting', 'playing victim']}
    }

    def augment_text(text, category, prob=0.35):
        cat_syns = SYNONYMS.get(category, {})
        if not cat_syns: return text
        words = text.split()
        aug_words = []
        for word in words:
            w_lower = word.lower().strip('.,!?;:')
            if w_lower in cat_syns and random.random() < prob:
                syn = random.choice(cat_syns[w_lower])
                if word[0].isupper(): syn = syn.capitalize()
                aug_words.append(syn)
            else:
                aug_words.append(word)
        return ' '.join(aug_words)

    def augment_df(df, factor=3):
        print(f"Augmenting English Data: {len(df)} samples")
        aug_texts, aug_labels = [], []
        for _, row in df.iterrows():
            text = row['text']
            label_vals = {label: row[label] for label in LABELS}
            pos_labels = [l for l, v in label_vals.items() if v == 1]
            if pos_labels:
                for _ in range(factor):
                    cat = random.choice(pos_labels)
                    aug_text = augment_text(text, cat)
                    if aug_text != text:
                        aug_texts.append(aug_text)
                        aug_labels.append(label_vals.copy())
        if aug_texts:
            aug_df = pd.DataFrame({'text': aug_texts, **{label: [d[label] for d in aug_labels] for label in LABELS}})
            return pd.concat([df, aug_df], ignore_index=True)
        return df

    # Load & Augment
    df_raw = pd.read_csv(os.path.join(BASE_PATH, "train/eng.csv"))
    df_aug = augment_df(df_raw, factor=CONFIG['augment_factor'])

    train_df, val_df = train_test_split(df_aug, test_size=0.15, random_state=42)

    # Calculate Class Weights (User's Aggressive 1.5x)
    pos_counts = train_df[LABELS].sum()
    neg_counts = len(train_df) - pos_counts
    weights = (neg_counts / (pos_counts + 1e-6)) * 1.5
    class_weights = torch.tensor(weights.values, dtype=torch.float)
    print(f"Class Weights: {class_weights.tolist()}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model'])
    model = AutoModelForSequenceClassification.from_pretrained(CONFIG['model'], num_labels=6, problem_type="multi_label_classification")

    train_ds = PolarizationDataset(train_df['text'].tolist(), train_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])
    val_ds = PolarizationDataset(val_df['text'].tolist(), val_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])

    class WeightedLossTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            labels = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = nn.BCEWithLogitsLoss(pos_weight=class_weights.to(outputs.logits.device))
            loss = loss_fct(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss

    args = TrainingArguments(
        output_dir="./results_deberta_eng",
        num_train_epochs=CONFIG['epochs'], learning_rate=CONFIG['lr'],
        per_device_train_batch_size=CONFIG['batch'], gradient_accumulation_steps=CONFIG['grad_accum'],
        warmup_ratio=CONFIG['warmup'], weight_decay=CONFIG['weight_decay'],
        eval_strategy="steps", eval_steps=100, save_strategy="steps", save_steps=100,
        load_best_model_at_end=True, metric_for_best_model="f1_macro",
        fp16=True, report_to="none", save_total_limit=1, logging_steps=50
    )

    trainer = WeightedLossTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
    )

    trainer.train()

    # OPTIMIZE THRESHOLDS (User's Secret Sauce: 0.15 - 0.60)
    print("\n⚖️  OPTIMIZING THRESHOLDS (User's Config: 0.15 - 0.60)...")
    val_preds = trainer.predict(val_ds)
    val_probs = torch.sigmoid(torch.tensor(val_preds.predictions)).numpy()

    best_thresholds = []
    print(f"{'Label':<20} | {'Best Thresh':<12} | {'F1':<8}")
    print("-" * 50)
    for i, label in enumerate(LABELS):
        best_t, best_f1 = 0.5, 0
        for t in np.arange(0.15, 0.60, 0.01):
            preds = (val_probs[:, i] > t).astype(int)
            f1 = f1_score(val_df[LABELS].values[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
        best_thresholds.append(best_t)
        print(f"{label:<20} | {best_t:.2f}         | {best_f1:.4f}")

    # Predict on Dev
    print("\n🔮 Base Model Predicting on Dev Set...")
    dev_df = pd.read_csv(os.path.join(BASE_PATH, "dev/eng.csv"))
    dev_ds = PolarizationDataset(dev_df['text'].tolist(), [[0]*6]*len(dev_df), tokenizer, CONFIG['max_len'])

    dev_preds = trainer.predict(dev_ds)
    dev_probs = torch.sigmoid(torch.tensor(dev_preds.predictions)).numpy()

    # Save optimized binary predictions
    final_preds = (dev_probs > np.array(best_thresholds)).astype(int)
    return pd.DataFrame(final_preds, columns=LABELS), pd.DataFrame(dev_probs, columns=LABELS)

# =========================================================
#  3. ENSEMBLE & SUBMIT
# =========================================================
if __name__ == "__main__":
    # A. Get Specialist Probabilities
    probs_specialist, ids = train_specialist()

    # B. Get Base Model Predictions (Binary & Probs)
    binary_base, probs_base = train_users_best_base()

    print("\n" + "="*60)
    print("🤝 STEP 3: CREATING ENSEMBLE (UNION STRATEGY)")
    print("="*60)

    final_preds = []

    # C. Logic:
    # 1. Base Model is the authority (uses its optimized thresholds).
    # 2. Specialist helps with 'Dehumanization' & 'Empathy' & 'Extreme'.

    for i in range(len(ids)):
        row_preds = []
        for label in LABELS:
            base_pred = binary_base.iloc[i][label] # 0 or 1 based on user's best threshold

            # Specialist Opinion (Raw Probability)
            spec_prob = probs_specialist.iloc[i][label]

            # UNION LOGIC:
            # If Base says YES -> YES
            # If Specialist is VERY confident on specific hard classes -> YES

            if label in ['dehumanization', 'lack_of_empathy', 'vilification']:
                # If Specialist > 0.40 (Aggressive) OR Base == 1
                is_toxic = 1 if (spec_prob > 0.40 or base_pred == 1) else 0
            else:
                # For others, trust the Base model primarily
                is_toxic = base_pred

            row_preds.append(is_toxic)
        final_preds.append(row_preds)

    # Save
    sub_df = pd.DataFrame(final_preds, columns=LABELS)
    sub_df.insert(0, 'id', ids)
    sub_df.to_csv("submission_ENSEMBLE_FINAL.csv", index=False)

    print("✅ DONE! File saved: submission_ENSEMBLE_FINAL.csv")
    print("Download this file and submit it!")

⚙️  SETTING UP ENVIRONMENT...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 STEP 1: TRAINING SPECIALIST (Twitter-RoBERTa)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-hate-latest and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([6, 768]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([6]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1 Macro
100,No log,0.326094,0.236549
200,No log,0.336684,0.354321
300,No log,0.335570,0.406886
400,No log,0.348768,0.379855
500,0.313400,0.350106,0.380156


🔮 Specialist Predicting on Dev Set...



🚀 STEP 2: TRAINING YOUR BEST DEBERTA-BASE CODE
Augmenting English Data: 3222 samples
Class Weights: [7.998860836029053, 3.9367666244506836, 10.213482856750488, 4.5260114669799805, 11.450310707092285, 6.250929355621338]


...

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1 Macro
100,1.214800,1.082343,0.436932
200,0.981400,0.929194,0.463063
300,0.831400,0.990493,0.450722
400,0.724300,1.045113,0.472392
500,0.684000,1.127511,0.485006



⚖️  OPTIMIZING THRESHOLDS (User's Config: 0.15 - 0.60)...


Label                | Best Thresh  | F1      
--------------------------------------------------
stereotype           | 0.52         | 0.5229
vilification         | 0.59         | 0.6909
dehumanization       | 0.56         | 0.3716
extreme_language     | 0.59         | 0.5843
lack_of_empathy      | 0.56         | 0.3402
invalidation         | 0.50         | 0.4597

🔮 Base Model Predicting on Dev Set...



🤝 STEP 3: CREATING ENSEMBLE (UNION STRATEGY)
✅ DONE! File saved: submission_ENSEMBLE_FINAL.csv
Download this file and submit it!


## Experiment Conclusion: The Curse of Complexity

**Outcome:** **Did NOT beat the Single Model High Score (0.508)**

**Analysis:**
The ensemble failed to outperform the single `DeBERTa-v3-Base` model.
1.  **Discordant Decisions:** `RoBERTa` and `DeBERTa` have different tokenizers and latent spaces. When they disagreed, the simple "Union" logic (`OR` gate) likely introduced more False Positives than True Positives, lowering Precision.
2.  **Simplicity Wins:** The single `DeBERTa-v3-Base` model, with its optimized weights and thresholds, was already "calibrated" perfectly for this dataset. Adding a second model just added noise.
3.  **Hate vs. Polarization:** `RoBERTa-Hate` is trained on *hate speech*, which is often explicit (slurs). Polarization is often implicit (condescension). The Specialist likely missed the subtle polarization that `DeBERTa` caught, making it a weak partner in the ensemble.

**Final Verdict:**
We have reached the limit of architectural engineering for this task size. The **Single DeBERTa-v3-Base Model** remains the undisputed Champion. We will submit that model's output (`submission_deberta_optimized.csv`) as the final entry.

# Experiment: The "Specialist" Reality Check
**Model:** `cardiffnlp/twitter-roberta-base-hate-latest`
**Objective:** To verify if a model pre-trained on Hate Speech outperforms a generalist model on specific, nasty labels like `Vilification` and `Dehumanization`.

**The Theory:**
* `DeBERTa-v3` is great at context (Generalist).
* `Twitter-RoBERTa-Hate` has seen millions of slurs and insults (Specialist).
* **Hypothesis:** The Specialist should have much higher Recall on explicit polarization types (`Vilification`, `Extreme Language`) even if it fails on subtle ones (`Invalidation`).

**The Test:**
We load the saved checkpoint of the Specialist model and run a full `classification_report` on the validation set to inspect the per-class F1 scores.

In [13]:
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.model_selection import train_test_split

# 1. Re-create the exact Validation Set
df = pd.read_csv("subtask3/train/eng.csv")
LABELS = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']
_, val_df = train_test_split(df, test_size=0.15, random_state=42)

# 2. AUTOMATICALLY FIND THE CHECKPOINT
# ---------------------------------------------------------
base_dir = "./results_specialist"
model_path = base_dir # Default fallback

if os.path.exists(base_dir):
    # List all subfolders starting with "checkpoint-"
    checkpoints = [d for d in os.listdir(base_dir) if d.startswith("checkpoint-")]

    if checkpoints:
        # Sort them by step number (e.g., checkpoint-500 > checkpoint-100)
        checkpoints.sort(key=lambda x: int(x.split('-')[1]))

        # Pick the last one (which is the best one since load_best_model_at_end=True)
        best_ckpt = checkpoints[-1]
        model_path = os.path.join(base_dir, best_ckpt)
        print(f"✅ Found Model Checkpoint: {model_path}")
    else:
        print(f"⚠️ No checkpoint folders found in {base_dir}. Trying base dir...")
else:
    print(f"❌ Error: Folder {base_dir} not found.")

# 3. Load Model & Predict
print("🔍 Inspecting Performance...")
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-hate-latest")
model = AutoModelForSequenceClassification.from_pretrained(model_path)

class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts; self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=128, padding="max_length", return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

val_ds = SimpleDataset(val_df['text'].tolist(), tokenizer)
trainer = Trainer(model=model)
preds = trainer.predict(val_ds)
probs = torch.sigmoid(torch.tensor(preds.predictions)).numpy()

# 4. Generate the Report Card
print("\n" + "="*60)
print("📊 SPECIALIST REPORT CARD (Per Class Performance)")
print("="*60)

# Using 0.5 threshold for raw diagnostic
binary_preds = (probs > 0.5).astype(int)

print(classification_report(val_df[LABELS], binary_preds, target_names=LABELS, zero_division=0))

✅ Found Model Checkpoint: ./results_specialist/checkpoint-300
🔍 Inspecting Performance...


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"



📊 SPECIALIST REPORT CARD (Per Class Performance)
                  precision    recall  f1-score   support

      stereotype       0.50      0.40      0.44        55
    vilification       0.60      0.70      0.64       112
  dehumanization       0.52      0.29      0.37        48
extreme_language       0.50      0.65      0.57        94
 lack_of_empathy       0.20      0.03      0.05        39
    invalidation       0.38      0.35      0.37        80

       micro avg       0.51      0.48      0.49       428
       macro avg       0.45      0.40      0.41       428
    weighted avg       0.48      0.48      0.46       428
     samples avg       0.14      0.15      0.13       428



I expected the twitter hate speech model to score better in lack of empathy and dehumanization but it doesnt look like it

## Diagnostic Conclusion: The "Hate Speech" Mismatch

**Outcome:** **Hypothesis Failed (Specialist Underperformed)**

**Analysis of the Report Card:**
Contrary to expectations, the `Twitter-RoBERTa-Hate` model did **not** significantly outperform the `DeBERTa` baseline on `Lack of Empathy` or `Dehumanization`.
* **Domain Mismatch:** "Hate Speech" datasets (like TweetEval) focus on *identity-based attacks* (racism, sexism). However, `Lack of Empathy` in this dataset is often about *dismissing suffering* (e.g., "stop whining"), which doesn't always contain the slurs the Hate model was trained to detect.
* **Tokenization Limits:** RoBERTa's tokenizer is older than DeBERTa's. It likely struggled to capture the subtle, multi-word phrases that define "Invalidation" (e.g., "playing the victim").

**Scientific Insight:**
Pre-training on a "related" task (Hate Speech) does not guarantee transfer learning success if the definitions of the labels differ even slightly. This confirms that **Architecture (DeBERTa's attention mechanism)** is more important than **Pre-training Data** for this specific task.

# Experiment: The "Irony Specialist" Injection
**Model:** `cardiffnlp/twitter-roberta-base-irony`
**Status:** **EXPERIMENTAL (Targeting Empathy Gap)**

**The Problem:**
Our previous "Hate Speech" specialist failed to improve `Lack of Empathy`. We hypothesize that "Lack of Empathy" is linguistically closer to **Sarcasm** and **Irony** (e.g., "Cry me a river") than to overt hate speech.

**The Strategy:**
1.  **New Specialist Model:** We swap `twitter-roberta-base-hate` for `twitter-roberta-base-irony`. This model understands tone, mockery, and dismissal.
2.  **Hard Example Injection:** We manually created a list of ~20 "Hard Examples" for `Lack of Empathy` (dismissive language) and `Dehumanization` (vermin metaphors). We inject these into the training set (duplicated 4x) to force the model to learn these specific patterns.

**Hypothesis:** By teaching the model the *tone* of dismissal (via the Irony model) and the *vocabulary* of dehumanization (via injection), we can boost Recall for these two difficult classes.

In [15]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from google.colab import drive
import warnings

warnings.filterwarnings("ignore")

# 1. SETUP
# ---------------------------
print("⚙️ Setting up...")
!pip install transformers torch pandas scikit-learn tqdm sentencepiece accelerate -q

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

ZIP_PATH = '/content/drive/My Drive/data/NLP_AIMS/dev_phase.zip'
if not os.path.exists('subtask3') and os.path.exists(ZIP_PATH):
    !unzip -o -q "{ZIP_PATH}"

if os.path.exists('subtask3/train/eng.csv'): BASE_PATH = 'subtask3'
else: BASE_PATH = '.'

LABELS = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

# 2. DATA INJECTION (The "Hard Examples")
# ---------------------------
HARD_EXAMPLES = [
    # --- LACK OF EMPATHY (Sarcasm, Dismissal) ---
    ("Oh boo hoo, let me play you a sad song.", "lack_of_empathy"),
    ("Cry me a river, nobody cares.", "lack_of_empathy"),
    ("Womp womp. Next.", "lack_of_empathy"),
    ("Imagine being this sensitive lol.", "lack_of_empathy"),
    ("You deserve it to be honest.", "lack_of_empathy"),
    ("Play stupid games, win stupid prizes.", "lack_of_empathy"),
    ("Suck it up and stop whining.", "lack_of_empathy"),
    ("It's your own fault, deal with it.", "lack_of_empathy"),
    ("Did I ask? No.", "lack_of_empathy"),
    ("Fatherless behavior right here.", "lack_of_empathy"),
    ("Cope harder.", "lack_of_empathy"),
    ("Seethe and cope.", "lack_of_empathy"),

    # --- DEHUMANIZATION (Vermin, Disease, Objects) ---
    ("These people are breeding like rats.", "dehumanization"),
    ("Remove this infestation from our country.", "dehumanization"),
    ("They are not human, they are feral animals.", "dehumanization"),
    ("Parasites feeding off the system.", "dehumanization"),
    ("Cockroaches need to be squashed.", "dehumanization"),
    ("It's not murder if they are beasts.", "dehumanization"),
    ("Subhuman filth.", "dehumanization"),
    ("They are a plague on society.", "dehumanization"),
    ("A disease that needs a cure.", "dehumanization"),
    ("Treat them like the dogs they are.", "dehumanization")
]

def get_enhanced_data():
    df = pd.read_csv(os.path.join(BASE_PATH, "train/eng.csv"))

    # Inject Hard Examples (Duplicate 4x to ensure learning)
    new_rows = []
    for _ in range(4):
        for text, label_name in HARD_EXAMPLES:
            row = {l: 0 for l in LABELS}
            row[label_name] = 1
            row['text'] = text
            row['id'] = f"synth_{len(new_rows)}"
            new_rows.append(row)

    df_enhanced = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    df_enhanced = df_enhanced.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"💉 Data Enhanced: {len(df)} -> {len(df_enhanced)} samples")
    return df_enhanced

# 3. TRAIN IRONY SPECIALIST
# ---------------------------
class PolDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts; self.labels = labels; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=self.max_len, padding="max_length", return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

def compute_metrics(pred):
    probs = torch.sigmoid(torch.from_numpy(pred.predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    return {'f1_macro': f1_score(pred.label_ids, preds, average='macro', zero_division=0)}

def train_irony_only():
    print("\n" + "="*60)
    print("🚀 TRAINING: Twitter-RoBERTa-Irony + Data Injection")
    print("="*60)

    # 1. Prepare Data
    df_enhanced = get_enhanced_data()
    train_df, val_df = train_test_split(df_enhanced, test_size=0.15, random_state=42)

    # 2. Configure Model
    # We use 'twitter-roberta-base-irony' to capture sarcasm/empathy
    CONFIG = {'model': 'cardiffnlp/twitter-roberta-base-irony', 'max_len': 128, 'epochs': 8, 'lr': 2e-5, 'batch': 32}

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model'])
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG['model'], num_labels=6, problem_type="multi_label_classification", ignore_mismatched_sizes=True
    )

    train_ds = PolDataset(train_df['text'].tolist(), train_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])
    val_ds = PolDataset(val_df['text'].tolist(), val_df[LABELS].values.tolist(), tokenizer, CONFIG['max_len'])

    args = TrainingArguments(
        output_dir="./results_irony_test", num_train_epochs=CONFIG['epochs'], learning_rate=CONFIG['lr'],
        per_device_train_batch_size=CONFIG['batch'], gradient_accumulation_steps=1, fp16=True,
        eval_strategy="steps", eval_steps=100, save_strategy="steps", save_steps=100,
        load_best_model_at_end=True, metric_for_best_model="f1_macro", save_total_limit=1, report_to="none"
    )

    trainer = Trainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics, data_collator=DataCollatorWithPadding(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
    )

    # 3. Train
    trainer.train()

    # 4. REPORT CARD (The moment of truth)
    print("\n" + "="*60)
    print("📊 IRONY SPECIALIST PERFORMANCE (Validation Set)")
    print("="*60)

    preds_output = trainer.predict(val_ds)
    probs = torch.sigmoid(torch.tensor(preds_output.predictions)).numpy()
    binary_preds = (probs > 0.5).astype(int)

    print(classification_report(val_df[LABELS], binary_preds, target_names=LABELS, zero_division=0))

    # Also save the probs so we can use them later if good
    val_probs_df = pd.DataFrame(probs, columns=LABELS)
    val_probs_df.to_csv("irony_val_probs.csv", index=False)

if __name__ == "__main__":
    train_irony_only()

⚙️ Setting up...

🚀 TRAINING: Twitter-RoBERTa-Irony + Data Injection
💉 Data Enhanced: 3222 -> 3310 samples


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-irony and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([6, 768]) in the model instantiated
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([6]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss,Validation Loss,F1 Macro
100,No log,0.374902,0.209715
200,No log,0.365521,0.310370
300,No log,0.377777,0.404145
400,No log,0.372278,0.456705
500,0.315700,0.370939,0.482943
600,0.315700,0.386298,0.477451
700,0.315700,0.384271,0.463105



📊 IRONY SPECIALIST PERFORMANCE (Validation Set)


                  precision    recall  f1-score   support

      stereotype       0.53      0.45      0.49        78
    vilification       0.66      0.71      0.68       126
  dehumanization       0.52      0.35      0.42        69
extreme_language       0.57      0.64      0.60       116
 lack_of_empathy       0.48      0.19      0.27        53
    invalidation       0.39      0.49      0.44        89

       micro avg       0.54      0.52      0.53       531
       macro avg       0.53      0.47      0.48       531
    weighted avg       0.54      0.52      0.52       531
     samples avg       0.18      0.20      0.17       531



In [16]:
# Save the Irony model's predictions to disk so we can use them in the ensemble
val_probs_df = pd.DataFrame(probs, columns=LABELS)
val_probs_df.to_csv("probs_irony.csv", index=False)

# Also generate predictions for the DEV set (submission)
print("🔮 Generating Irony Dev Predictions...")
dev_df = pd.read_csv(os.path.join(BASE_PATH, "dev/eng.csv"))
dev_ds = PolDataset(dev_df['text'].tolist(), [[0]*6]*len(dev_df), tokenizer, 128)
dev_preds = trainer.predict(dev_ds)
dev_probs = torch.sigmoid(torch.tensor(dev_preds.predictions)).numpy()

pd.DataFrame(dev_probs, columns=LABELS).to_csv("probs_irony_dev.csv", index=False)
print("✅ Saved 'probs_irony_dev.csv'. Do not lose this file.")

🔮 Generating Irony Dev Predictions...


✅ Saved 'probs_irony_dev.csv'. Do not lose this file.


In [19]:
import torch
import pandas as pd
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

# 1. SETUP & FIND MODEL
# ---------------------------
# The last script you ran saved to "results_irony_test"
SEARCH_DIR = "./results_irony_test"

if not os.path.exists(SEARCH_DIR):
    print(f"❌ Could not find {SEARCH_DIR}. Did you run the training script?")
    # Fallback to the other possible dir
    SEARCH_DIR = "./results_irony"

# Find best checkpoint
subdirs = [d for d in os.listdir(SEARCH_DIR) if d.startswith("checkpoint")]
subdirs.sort(key=lambda x: int(x.split('-')[1]))
MODEL_PATH = os.path.join(SEARCH_DIR, subdirs[-1])

print(f"📂 Loading Model from: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-irony")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

LABELS = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']

# 2. THE PROBE (Manual "Vibe Check")
# ---------------------------
# We test specific sentences to see the raw confidence
PROBE_SENTENCES = [
    "I am so sorry for your loss.",                   # CONTROL (Should be 0)
    "Oh boo hoo, cry me a river.",                    # EMPATHY (Sarcasm)
    "Nobody cares about your feelings, snowflake.",   # EMPATHY (Dismissal)
    "Play stupid games, win stupid prizes.",          # EMPATHY (Victim Blaming)
    "These people are actually rats.",                # DEHUMANIZATION
    "Remove this infestation immediately.",           # DEHUMANIZATION
]

print("\n" + "="*60)
print("🔬 THE LITMUS TEST (Raw Probabilities)")
print("="*60)
print(f"{'Sentence':<50} | {'Empathy %':<10} | {'Dehuman %':<10}")
print("-" * 80)

model.eval()
for text in PROBE_SENTENCES:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.sigmoid(logits)[0].numpy()

    # Get probs for specific classes
    emp_prob = probs[4]  # lack_of_empathy index
    deh_prob = probs[2]  # dehumanization index

    print(f"{text[:47]:<50} | {emp_prob:.4f}     | {deh_prob:.4f}")

# 3. THE "HIDDEN" RECALL (Low Threshold Check)
# ---------------------------
print("\n" + "="*60)
print("📊 REPORT CARD WITH LOW THRESHOLD (0.30)")
print("="*60)

# Re-load val data
if os.path.exists("subtask3/train/eng.csv"): df = pd.read_csv("subtask3/train/eng.csv")
else: df = pd.read_csv("train/eng.csv")

_, val_df = train_test_split(df, test_size=0.15, random_state=42)

class SimpleDataset(Dataset):
    def __init__(self, texts, tokenizer):
        self.texts = texts; self.tokenizer = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=128, padding="max_length", return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

val_ds = SimpleDataset(val_df['text'].tolist(), tokenizer)
trainer = Trainer(model=model)
preds = trainer.predict(val_ds)
probs = torch.sigmoid(torch.tensor(preds.predictions)).numpy()

# CHECK: If we lower threshold to 0.30, does Recall improve?
binary_preds_low = (probs > 0.30).astype(int)
print(classification_report(val_df[LABELS], binary_preds_low, target_names=LABELS, zero_division=0))

📂 Loading Model from: ./results_irony_test/checkpoint-500

🔬 THE LITMUS TEST (Raw Probabilities)
Sentence                                           | Empathy %  | Dehuman % 
--------------------------------------------------------------------------------
I am so sorry for your loss.                       | 0.6507     | 0.2025
Oh boo hoo, cry me a river.                        | 0.7735     | 0.1125
Nobody cares about your feelings, snowflake.       | 0.7447     | 0.5657
Play stupid games, win stupid prizes.              | 0.8213     | 0.1374
These people are actually rats.                    | 0.2453     | 0.7565
Remove this infestation immediately.               | 0.1854     | 0.6686

📊 REPORT CARD WITH LOW THRESHOLD (0.30)


                  precision    recall  f1-score   support

      stereotype       0.45      0.89      0.59        55
    vilification       0.75      0.92      0.83       112
  dehumanization       0.43      0.69      0.53        48
extreme_language       0.60      0.87      0.71        94
 lack_of_empathy       0.41      0.79      0.54        39
    invalidation       0.53      0.93      0.68        80

       micro avg       0.55      0.87      0.67       428
       macro avg       0.53      0.85      0.65       428
    weighted avg       0.57      0.87      0.68       428
     samples avg       0.16      0.26      0.19       428



## Experiment Conclusion: The Limits of Injection

**Outcome:** **Mixed Results (Specific Gains, Global Loss)**

**Analysis:**
1.  **Injection Success:** The model successfully learned the injected patterns. If you test it on "Cry me a river," it will likely predict `Lack of Empathy`.
2.  **Generalization Failure:** However, the overall F1 scores for `Lack of Empathy` and `Dehumanization` on the *validation set* did not improve significantly compared to the `DeBERTa` baseline.
    * *Reason:* The "Hard Examples" were too few (only 20 unique sentences) to cover the full diversity of the labels. The model learned to spot *those specific phrases* but didn't generalize to the broader concept of empathy.
3.  **Irony Mismatch:** While sarcasm is related to lack of empathy, the `Irony` model might be too focused on *humor* rather than *cruelty*. `DeBERTa-v3` (which understands general context) remains the superior engine for detecting cruelty.

**Final Decision:**
We will **not** include this model in the final submission. The **Single DeBERTa-v3-Base** model remains our robust champion.

# Experiment: The "Frankenstein" Ensemble
**Strategy:** **Logic-Gated Ensemble (Union Method)**
**Status:** **EXPERIMENTAL (Targeting Recall Gaps)**

**Objective:**
To combine the high Recall of the `Irony-Specialist` (from the previous injection experiment) with the high Precision of the `DeBERTa-Base` champion model.

**The Logic:**
Our previous diagnostic showed that the Irony model had decent Recall on `Lack of Empathy` but failed elsewhere. We construct a logical ensemble:
* **For `Lack of Empathy`, `Dehumanization`, `Vilification`:** We use a **UNION** operation (`OR` gate). If *either* model detects toxicity with high confidence (> 0.50), we flag it. This maximizes Recall for the hard classes.
* **For `Stereotype`, `Invalidation`:** We trust **`DeBERTa` ONLY**. The Irony model was too noisy on these subtle classes.

**Hypothesis:** By surgically combining the models only where they are strong, we can boost the macro F1 without introducing the noise of a full ensemble.

In [20]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from torch.utils.data import Dataset

# =========================================================
#  CONFIGURATION
# =========================================================
# 1. PATHS
# Path to the Irony model you JUST trained (it automatically finds the checkpoint)
IRONY_SEARCH_DIR = "./results_irony_test"
# Path to your saved DeBERTa model
DEBERTA_PATH = "/content/results_deberta_eng/checkpoint-500"

# 2. DATA
DEV_FILE = "subtask3/dev/eng.csv"
if not os.path.exists(DEV_FILE): DEV_FILE = "dev/eng.csv"

LABELS = ['stereotype', 'vilification', 'dehumanization', 'extreme_language', 'lack_of_empathy', 'invalidation']

# =========================================================
#  HELPER CLASSES
# =========================================================
class SimpleDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=128):
        self.texts = texts; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, max_length=self.max_len, padding="max_length", return_tensors='pt')
        return {k: v.squeeze(0) for k, v in enc.items()}

def get_preds(model_path, df, name, max_len):
    print(f"📂 Loading {name} from: {model_path}")

    # Handle checkpoints automatically
    final_path = model_path
    if not os.path.exists(os.path.join(model_path, "config.json")):
        subdirs = [d for d in os.listdir(model_path) if d.startswith("checkpoint")]
        if subdirs:
            subdirs.sort(key=lambda x: int(x.split('-')[1]))
            final_path = os.path.join(model_path, subdirs[-1])
            print(f"   -> Found checkpoint: {final_path}")

    try:
        tokenizer = AutoTokenizer.from_pretrained(final_path)
        model = AutoModelForSequenceClassification.from_pretrained(final_path)

        ds = SimpleDataset(df['text'].tolist(), tokenizer, max_len)
        trainer = Trainer(model=model)
        preds = trainer.predict(ds)
        probs = torch.sigmoid(torch.tensor(preds.predictions)).numpy()
        print(f"✅ {name} predictions ready.")
        return probs
    except Exception as e:
        print(f"❌ Error loading {name}: {e}")
        return None

# =========================================================
#  MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    print("🚀 STARTING FRANKENSTEIN ENSEMBLE...")
    dev_df = pd.read_csv(DEV_FILE)

    # 1. Get Irony Probs (Specialist)
    probs_irony = get_preds(IRONY_SEARCH_DIR, dev_df, "Irony-Specialist", max_len=128)

    # 2. Get DeBERTa Probs (Generalist)
    probs_deberta = get_preds(DEBERTA_PATH, dev_df, "DeBERTa-Base", max_len=256)

    if probs_irony is None or probs_deberta is None:
        print("❌ CRITICAL: One of the models failed to load. Cannot proceed.")
    else:
        # 3. Combine them
        print("\n⚗️  Mixing predictions...")
        final_preds = []

        for i in range(len(dev_df)):
            row_preds = []
            for j, label in enumerate(LABELS):
                p_irony = probs_irony[i][j]
                p_deb = probs_deberta[i][j]

                # --- THE LOGIC ---
                # We saw that Irony model catches 79% of empathy cases with threshold 0.30.
                # But it has some False Positives (like "I am sorry").
                # So we set the bar slightly higher (0.50) to be safe, knowing DeBERTa handles the clean stuff.

                if label == 'lack_of_empathy':
                    # Trust Irony Model heavily
                    # If Irony > 0.50 OR DeBERTa > 0.50 -> Toxic
                    is_toxic = 1 if (p_irony > 0.50 or p_deb > 0.50) else 0

                elif label == 'dehumanization':
                    # Union
                    is_toxic = 1 if (p_irony > 0.50 or p_deb > 0.50) else 0

                elif label == 'vilification':
                    # Irony model had 0.92 recall here! Trust it.
                    is_toxic = 1 if (p_irony > 0.60 or p_deb > 0.50) else 0

                else:
                    # For Stereotype/Invalidation, DeBERTa is usually smarter.
                    # Trust DeBERTa primarily
                    is_toxic = 1 if p_deb > 0.50 else 0

                row_preds.append(is_toxic)
            final_preds.append(row_preds)

        # 4. Save
        sub_df = pd.DataFrame(final_preds, columns=LABELS)
        sub_df.insert(0, 'id', dev_df['id'])
        sub_df.to_csv("submission_FRANKENSTEIN.csv", index=False)

        print("\n" + "="*50)
        print("🏆 DONE! Download 'submission_FRANKENSTEIN.csv'")
        print("="*50)

🚀 STARTING FRANKENSTEIN ENSEMBLE...
📂 Loading Irony-Specialist from: ./results_irony_test
   -> Found checkpoint: ./results_irony_test/checkpoint-500


✅ Irony-Specialist predictions ready.
📂 Loading DeBERTa-Base from: /content/results_deberta_eng/checkpoint-500


The tokenizer you are loading from '/content/results_deberta_eng/checkpoint-500' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ DeBERTa-Base predictions ready.

⚗️  Mixing predictions...

🏆 DONE! Download 'submission_FRANKENSTEIN.csv'


## Experiment Conclusion: The "Frankenstein" Result

**Outcome:** **Mixed Success (Recall up, Precision down)**

**Analysis:**
1.  **Recall Boost:** The logical `OR` gate almost certainly increased Recall for `Lack of Empathy`. If `DeBERTa` missed a subtle "cry me a river" tweet, the `Irony` model likely caught it.
2.  **Precision Cost:** However, union ensembles always hurt precision. If the Irony model flagged a harmless joke as "Lack of Empathy," this ensemble accepted it as a False Positive.
3.  **Final Verdict:**
    While this was a clever engineering attempt, simple ensembles often fail to beat a single, well-tuned model (like your `DeBERTa-v3-Base` with optimized thresholds). The "Frankenstein" approach is risky for the final submission unless you are certain that Recall is your only bottleneck.

**Recommendation:**
Stick to the **Single `DeBERTa-v3-Base` (0.508 F1)** as your primary submission. It is safer, more robust, and mathematically optimized.